In [1]:
# Cell 1: Setup and imports
import sys
import pandas as pd
import numpy as np
import os
from pathlib import Path
import importlib

# Add src to Python path using an absolute project path
sys.path.insert(0, str(Path('..').resolve()))

# Clear stale imports if the notebook kernel already loaded src before
if 'src.config' in sys.modules:
    del sys.modules['src.config']
if 'src' in sys.modules:
    del sys.modules['src']

config = importlib.import_module('src.config')
importlib.reload(config)

RAW_DATA_DIR = config.RAW_DATA_DIR
PROCESSED_DATA_DIR = config.PROCESSED_DATA_DIR
DHS_FILE_MAPPING = config.DHS_FILE_MAPPING
KEY_VARIABLES = config.KEY_VARIABLES

from src.data_loader import load_dhs_file, load_all_ethiopia_dhs, get_variable_info

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Files found: {list(RAW_DATA_DIR.glob('*.DTA'))}")


Raw data directory: C:\Users\hp\Desktop\ethiopia-early-marriage-dhs\data\raw
Files found: [WindowsPath('C:/Users/hp/Desktop/ethiopia-early-marriage-dhs/data/raw/ETIR41FL.DTA'), WindowsPath('C:/Users/hp/Desktop/ethiopia-early-marriage-dhs/data/raw/ETIR51FL.DTA'), WindowsPath('C:/Users/hp/Desktop/ethiopia-early-marriage-dhs/data/raw/ETIR61FL.DTA'), WindowsPath('C:/Users/hp/Desktop/ethiopia-early-marriage-dhs/data/raw/ETIR71FL.DTA')]


In [2]:
# Cell 2: Load all DHS files
all_data = load_all_ethiopia_dhs(RAW_DATA_DIR)

for year, data_dict in all_data.items():
    df = data_dict['df']
    metadata = data_dict['metadata']
    print(f"\n{'='*50}")
    print(f"Year {year}: {metadata['file_name']}")
    print(f"Rows: {metadata['n_rows']:,}")
    print(f"Columns: {metadata['n_columns']}")
    print(f"First 5 columns: {list(df.columns[:5])}")

2026-04-04 22:14:39,976 - INFO - Loading ETIR41FL.DTA...
2026-04-04 22:14:51,794 - INFO - Successfully loaded 15,367 rows and 3665 columns
2026-04-04 22:14:51,800 - INFO - Loading ETIR51FL.DTA...
2026-04-04 22:15:04,633 - INFO - Successfully loaded 14,070 rows and 4699 columns
2026-04-04 22:15:04,637 - INFO - Loading ETIR61FL.DTA...
2026-04-04 22:15:16,095 - INFO - Successfully loaded 16,515 rows and 3807 columns
2026-04-04 22:15:16,099 - INFO - Loading ETIR71FL.DTA...
2026-04-04 22:15:34,256 - INFO - Successfully loaded 15,683 rows and 5902 columns



Year 2000: ETIR41FL.DTA
Rows: 15,367
Columns: 3665
First 5 columns: ['caseid', 'v000', 'v001', 'v002', 'v003']

Year 2005: ETIR51FL.DTA
Rows: 14,070
Columns: 4699
First 5 columns: ['caseid', 'v000', 'v001', 'v002', 'v003']

Year 2011: ETIR61FL.DTA
Rows: 16,515
Columns: 3807
First 5 columns: ['caseid', 'v000', 'v001', 'v002', 'v003']

Year 2016: ETIR71FL.DTA
Rows: 15,683
Columns: 5902
First 5 columns: ['caseid', 'v000', 'v001', 'v002', 'v003']


In [3]:
# Cell 3: Focus on year 2000 (ETIR41FL.DTA) - inspect key variables
df_2000 = all_data[2000]['df']

print("Key variables for early marriage study:\n")
for var_name, var_code in KEY_VARIABLES.items():
    if var_code in df_2000.columns:
        print(f"{var_name:25} -> {var_code}")
    else:
        print(f"{var_name:25} -> {var_code} [NOT FOUND]")

Key variables for early marriage study:

age_first_marriage        -> v511
marriage_cohort           -> v512
current_marital_status    -> v501
children_ever_born        -> v201
children_surviving        -> v202
current_contraceptive     -> v312
ideal_number_children     -> v613
education_level           -> v106
education_years           -> v107
literacy                  -> v155
current_work              -> v714
wealth_index              -> v190 [NOT FOUND]
media_exposure            -> v157
current_age               -> v012
region                    -> v024
residence                 -> v025
religion                  -> v130
ethnicity                 -> v131
household_head_sex        -> v151
water_source              -> v113
toilet_facility           -> v116
electricity               -> v119


In [12]:
# Find wealth-related columns in 2000 data
wealth_cols = [col for col in df_2000.columns if 'wealth' in col.lower() or 'v190' in col or 'v191' in col]
print("Wealth-related columns in 2000:", wealth_cols)

# Also check v190 directly
if 'v190' in df_2000.columns:
    print("v190 exists in 2000")
else:
    print("v190 NOT in 2000 - checking v191...")
    if 'v191' in df_2000.columns:
        print("v191 exists (wealth index factor score)")

Wealth-related columns in 2000: []
v190 NOT in 2000 - checking v191...


In [4]:
# Cell 4: Understand v511 (age at first marriage) - OUR MAIN OUTCOME
print("=== Age at First Marriage (v511) ===\n")
info = get_variable_info(df_2000, 'v511')
print(f"Data type: {info['dtype']}")
print(f"Missing values: {info['n_missing']:,}")
print(f"Unique values: {info['n_unique']}")
print(f"\nValue distribution (top 10):")
for value, count in info['unique_values'].items():
    print(f"  Age {int(value)}: {count:,} women")

=== Age at First Marriage (v511) ===

Data type: object
Missing values: 3,979
Unique values: 35

Value distribution (top 10):
  Age 15: 2,296 women
  Age 16: 1,403 women
  Age 14: 1,251 women
  Age 17: 999 women
  Age 18: 989 women
  Age 13: 749 women
  Age 12: 667 women
  Age 20: 630 women
  Age 19: 591 women
  Age 10: 421 women


In [5]:
# Cell 5: Understand v201 (children ever born) - CONSEQUENCE
print("=== Children Ever Born (v201) ===\n")
info = get_variable_info(df_2000, 'v201')
print(f"Data type: {info['dtype']}")
print(f"Missing values: {info['n_missing']:,}")
print(f"Unique values: {info['n_unique']}")
print(f"\nValue distribution (top 10):")
for value, count in info['unique_values'].items():
    print(f"  {int(value)} children: {count:,} women")

=== Children Ever Born (v201) ===

Data type: int64
Missing values: 0
Unique values: 18

Value distribution (top 10):
  0 children: 5,224 women
  1 children: 1,765 women
  2 children: 1,596 women
  3 children: 1,312 women
  4 children: 1,107 women
  5 children: 1,046 women
  6 children: 995 women
  7 children: 771 women
  8 children: 636 women
  9 children: 403 women


In [6]:
# Cell 6: Check value labels for categorical variables
# DHS codes numeric values but has labels (e.g., 1=No education, 2=Primary, etc.)

def show_value_labels(df, variable):
    """Show value labels for a DHS categorical variable"""
    if variable in df.columns:
        print(f"\n{variable} - Value Labels:")
        # Get unique values and their frequencies
        value_counts = df[variable].value_counts().sort_index()
        for val, count in value_counts.items():
            print(f"  {val}: n={count:,}")
    else:
        print(f"{variable} not found")

# Check education variable (v106)
show_value_labels(df_2000, 'v106')

# Check residence (v025)
show_value_labels(df_2000, 'v025')

# Check wealth index (v190)
show_value_labels(df_2000, 'v190')


v106 - Value Labels:
  0: n=10,586
  1: n=2,530
  2: n=2,092
  3: n=159

v025 - Value Labels:
  1: n=4,543
  2: n=10,824
v190 not found


In [7]:
# Cell 7: Create early marriage indicator
# Define early marriage as age at first marriage < 18

df_2000['early_marriage'] = (df_2000['v511'] < 18) & (df_2000['v511'].notna())
early_marriage_rate = df_2000['early_marriage'].mean() * 100

print(f"Year 2000 - Early marriage rate (<18): {early_marriage_rate:.1f}%")
print(f"Total women: {len(df_2000):,}")
print(f"Married before 18: {df_2000['early_marriage'].sum():,}")

Year 2000 - Early marriage rate (<18): 52.4%
Total women: 15,367
Married before 18: 8,050


In [14]:
# More accurate early marriage rate (excluding never married)
married_women = df_2000[df_2000['v501'].isin([1,2])]  # 1=married, 2=living together
early_among_married = (married_women['v511'] < 18).mean() * 100
print(f"Early marriage rate among ever-married women: {early_among_married:.1f}%")

Early marriage rate among ever-married women: 69.9%


In [15]:
# Cell A: Check wealth variables in all years
for year, data_dict in all_data.items():
    df = data_dict['df']
    wealth_vars = [c for c in df.columns if c in ['v190', 'v191', 'v190a', 'wealth']]
    print(f"{year}: Wealth vars found = {wealth_vars}")

2000: Wealth vars found = []
2005: Wealth vars found = ['v190', 'v191']
2011: Wealth vars found = ['v190', 'v191']
2016: Wealth vars found = ['v190', 'v191', 'v190a']


In [17]:
# Cell B: Fix v511 data type and check distribution
df_2000['v511_numeric'] = pd.to_numeric(df_2000['v511'], errors='coerce')
print(f"After conversion - missing: {df_2000['v511_numeric'].isna().sum()}")
print(f"Age range: {df_2000['v511_numeric'].min():.0f} to {df_2000['v511_numeric'].max():.0f}")

After conversion - missing: 3979
Age range: 5 to 47


In [18]:
# Cell C: More accurate early marriage rate
never_married = df_2000[df_2000['v501'] == 0]
married_women = df_2000[df_2000['v501'].isin([1,2])]

print(f"Never married: {len(never_married):,} ({len(never_married)/len(df_2000)*100:.1f}%)")
print(f"Currently married/living together: {len(married_women):,}")
print(f"Early marriage rate among ever-married: {(married_women['v511_numeric'] < 18).mean()*100:.1f}%")

Never married: 3,979 (25.9%)
Currently married/living together: 9,380
Early marriage rate among ever-married: 69.9%


In [19]:
# Check variable availability across ALL years
def check_variable_availability(all_data, variables):
    availability = {}
    for var_name, var_code in variables.items():
        available_years = []
        for year, data_dict in all_data.items():
            df = data_dict['df']
            if var_code in df.columns:
                available_years.append(year)
        availability[var_name] = available_years
    return availability

# Run this to see what's actually available
availability = check_variable_availability(all_data, KEY_VARIABLES)
for var, years in availability.items():
    print(f"{var:25} -> Available: {years if years else 'NONE'}")

age_first_marriage        -> Available: [2000, 2005, 2011, 2016]
marriage_cohort           -> Available: [2000, 2005, 2011, 2016]
current_marital_status    -> Available: [2000, 2005, 2011, 2016]
children_ever_born        -> Available: [2000, 2005, 2011, 2016]
children_surviving        -> Available: [2000, 2005, 2011, 2016]
current_contraceptive     -> Available: [2000, 2005, 2011, 2016]
ideal_number_children     -> Available: [2000, 2005, 2011, 2016]
education_level           -> Available: [2000, 2005, 2011, 2016]
education_years           -> Available: [2000, 2005, 2011, 2016]
literacy                  -> Available: [2000, 2005, 2011, 2016]
current_work              -> Available: [2000, 2005, 2011, 2016]
wealth_index              -> Available: [2005, 2011, 2016]
media_exposure            -> Available: [2000, 2005, 2011, 2016]
current_age               -> Available: [2000, 2005, 2011, 2016]
region                    -> Available: [2000, 2005, 2011, 2016]
residence                 -> Av

In [21]:
# Quality checks for each candidate variable
def assess_quality(df, var_code):
    missing_pct = df[var_code].isna().mean() * 100
    unique_vals = df[var_code].nunique()
    return {
        'missing_pct': missing_pct,
        'unique_vals': unique_vals,
        'usable': missing_pct < 30  # Threshold
    }

In [22]:
# Updated variable dictionary with availability
REVISED_VARIABLES = {
    # OUTCOME VARIABLES
    'age_first_marriage': {'code': 'v511', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    
    # CONSEQUENCES
    'children_ever_born': {'code': 'v201', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'education_level': {'code': 'v106', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'current_work': {'code': 'v714', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    
    # DETERMINANTS
    'current_age': {'code': 'v012', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'residence': {'code': 'v025', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'region': {'code': 'v024', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'religion': {'code': 'v130', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    
    # WEALTH - PROBLEMATIC
    'wealth_index': {'code': 'v190', 'years': [2005,2011,2016], 'quality': 'missing_2000'},
    
    # ALTERNATIVE WEALTH PROXY FOR 2000
    'electricity': {'code': 'v119', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'water_source': {'code': 'v113', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'toilet': {'code': 'v116', 'years': [2000,2005,2011,2016], 'quality': 'good'},
    'floor_material': {'code': 'v127', 'years': [2000,2005,2011,2016], 'quality': 'check'},
}

In [23]:
# Complete availability check for ALL potential variables
all_potential_vars = {
    'v511': 'Age at first marriage',
    'v512': 'Year of marriage', 
    'v501': 'Marital status',
    'v201': 'Children ever born',
    'v106': 'Education level',
    'v107': 'Education years',
    'v155': 'Literacy',
    'v714': 'Working status',
    'v012': 'Current age',
    'v024': 'Region',
    'v025': 'Residence',
    'v130': 'Religion',
    'v131': 'Ethnicity',
    'v190': 'Wealth index',
    'v191': 'Wealth score',
    'v119': 'Electricity',
    'v113': 'Water source',
    'v116': 'Toilet',
    'v127': 'Floor material',
    'v151': 'HH head sex',
}

print("Variable Availability Across Survey Years")
print("=" * 60)

for var_code, var_desc in all_potential_vars.items():
    available = []
    for year, data_dict in all_data.items():
        df = data_dict['df']
        if var_code in df.columns:
            available.append(str(year))
    status = "✓ " + ", ".join(available) if available else "✗ NOT FOUND"
    print(f"{var_code:6} ({var_desc[:25]:25}) -> {status}")

Variable Availability Across Survey Years
v511   (Age at first marriage    ) -> ✓ 2000, 2005, 2011, 2016
v512   (Year of marriage         ) -> ✓ 2000, 2005, 2011, 2016
v501   (Marital status           ) -> ✓ 2000, 2005, 2011, 2016
v201   (Children ever born       ) -> ✓ 2000, 2005, 2011, 2016
v106   (Education level          ) -> ✓ 2000, 2005, 2011, 2016
v107   (Education years          ) -> ✓ 2000, 2005, 2011, 2016
v155   (Literacy                 ) -> ✓ 2000, 2005, 2011, 2016
v714   (Working status           ) -> ✓ 2000, 2005, 2011, 2016
v012   (Current age              ) -> ✓ 2000, 2005, 2011, 2016
v024   (Region                   ) -> ✓ 2000, 2005, 2011, 2016
v025   (Residence                ) -> ✓ 2000, 2005, 2011, 2016
v130   (Religion                 ) -> ✓ 2000, 2005, 2011, 2016
v131   (Ethnicity                ) -> ✓ 2000, 2005, 2011, 2016
v190   (Wealth index             ) -> ✓ 2005, 2011, 2016
v191   (Wealth score             ) -> ✓ 2005, 2011, 2016
v119   (Electricity      

# Key Variables

| Concept | DHS Code | Description | Notes |
|---------|----------|-------------|-------|
| Age at first marriage | v511 | Age in years at first marriage | Primary outcome variable. Values 0-49, 99=missing |
| Current age | v012 | Age in years | Demographic control |
| Children ever born | v201 | Total children ever born | Consequence measure |
| Education level | v106 | Highest education level | 0=No education, 1=Primary, 2=Secondary, 3=Higher |
| Wealth index | v190 | Wealth quintile | 1=Poorest to 5=Richest |
| Residence | v025 | Urban/rural | 1=Urban, 2=Rural |
| Region | v024 | Administrative region | 1=Tigray, 2=Afar, 3=Amhara, etc. |

## Early Marriage Definition
- **Threshold**: Age at first marriage < 18 years
- **Source**: UNICEF and DHS standard definition

## Data Files Loaded
"""

In [11]:
# Cell 8: Save variable dictionary as markdown for reports

variable_doc = f"""# DHS Variable Dictionary - Ethiopia Early Marriage Study

## Key Variables

| Concept | DHS Code | Description | Notes |
|---------|----------|-------------|-------|
| Age at first marriage | v511 | Age in years at first marriage | Primary outcome variable. Values 0-49, 99=missing |
| Current age | v012 | Age in years | Demographic control |
| Children ever born | v201 | Total children ever born | Consequence measure |
| Education level | v106 | Highest education level | 0=No education, 1=Primary, 2=Secondary, 3=Higher |
| Wealth index | v190 | Wealth quintile | 1=Poorest to 5=Richest |
| Residence | v025 | Urban/rural | 1=Urban, 2=Rural |
| Region | v024 | Administrative region | 1=Tigray, 2=Afar, 3=Amhara, etc. |

## Early Marriage Definition
- **Threshold**: Age at first marriage < 18 years
- **Source**: UNICEF and DHS standard definition

## Data Files Loaded
"""

for year in [2000, 2005, 2011, 2016]:
    variable_doc += f"- {year}: ETIR{['41','51','61','71'][year//5 - 400]}FL.DTA\n"

# Save to file
with open('../reports/variable_dictionary.md', 'w') as f:
    f.write(variable_doc)

print("Variable dictionary saved to reports/variable_dictionary.md")

Variable dictionary saved to reports/variable_dictionary.md
